# Objective
In this notebook we simulate telemetry emitted by a series of events generated by transactions passing through non-linear sequences of noisy processes with individually varying processing latency and service rates.  

Transactions are created at the root of processing trees, and generate events in subsequent processes in that same tree as they pass through.

The *first critical dependency* to being able to use such event data analytically is to be able to faithfully determine the processinging tree through which a transaction passed.  Current approaches in the marketplace require one or more of the following to achieve this:
- publishers to have knowledge of the processes upstream and/or downstream of them (or indeed the entire tree), and to publish this information with the event data or to configure it centrally within the collection process,
- publishers to use a common identifier accross all events generated by a transaction,
- more advanced approaches automatically inject UUIDs into message headers and rely on processes to pass these on to subsequent events in the tree.  This can be problematic when transactions run accross different technologies which in turn requires custom logic to maintain these UUIDs across the chain.

Here we take a different approach by attempting to abstract away as much need as possible for both the event publisher and subsequent consumers of the event date to know anything about other processes in the stack.  Our requirements are minimal:
- a process must generate an event at a consistent point in its processing logic (it doesn't matter where, just that it is consistent within that process)
- the event must be uniquely identifiable via an event name (and optionally other identifiers)
- the event must contain a transaction reference that is shared with at least one other event in the tree.  Our recommendation is that processes publish both their native transaction ID as well as one tranaction ID from an upstream process where this differs.


The *second critical dependency* to being able to use event data analytically is to preserve the granularity of the original observations.  The quantity of telemetry generated by a process is at least a factor if not several orders of magnitude in volume greater than its transaction data.  Think of a trasnaction that passes through 10 processes: in order to instrument each of those processes, 10 events will be generate for each transaction.  The general approach in the current marketplace is to downsample telemetry data however this approach precludes subsequent application of the vast majority of statistical techniques (such as t-tests, linear correlation, and fitting data to probability distributions) and limiting analysis to the summary statistics at the granularity at which data is now retained. 

Again our approach is contrary to trend in that we attempt to store original observations in such a way that support efficient retrieval.


Our assumption, commercially, is that by centralising this logic within the collecting processes and raising the level of abstraction for publishers and consumers, the overall cost of collecting and correlating event data is optimised where there are many publishers and a single collector.  This does of course require complex inference at scale on the part of the collector, and the exporation of that problems and possible solutions is what this notebook explores and sets out to prove as feasible.


## Our problem statements - the many unkowns
In such a system above, the collector tasked with correlating events will face the following challenges:

1. The rate at which transactions are created is stochastic, highly variable and the distribution is unknown in advance.  However, under the Central Limit Theorum we can assume that the means of samples from that distribution are normally distributed.  TODO: check this assumption with Dimitrios.

2. The number of nodes in the processing tree nor their adjacency is known in advance.  However it is possible to identify the exact process in which an event was generated via an explicit identifier.

3. The correlation of events into a causally related event chain/tree will only be possible deterministically via common transaction identifiers.  Each event will reference one or more identifiers with the sole constraint that each event must share an identifier with at least one other event in the chain.  There can be any number of transaction identifiers in an event chain.  This means that individual events in the chain may have mutually exclusive identifiers, and events and identifiers can be receieved in any order.

4. The time taken for a transaction to pass through a given process is stochastic, assumed to follow a normal distribution but the parameters of that distribution are unknown in advance.

5. The rate at which transactions are created, at times, can exceed the service rates of processes in the tree, such that a queue may form at the ingress of those processes.

6. Exceptionally, processes may exhibit anomalous behaviour, orthogpnal to the arrival rate of events to that process, this includes: a) anomalous processing latency, b) anomalous service rate, c) inaccurate data published to the telemetry collector.

7. A process need only generate a single event, although it is assumed that the point in the processing path at which events are generated is consistent.  Consequently the breakdown of latency between events due to #4 processing time, #5 queuing, #6 anomalous behaviour cannot be directly observed.

8.  Finally no assumptions can be made as to the sequence in which events are observed by our telemetry process: events from multiple differing event chains may be interleaved, events may be received out of order etc.

...phew.  Reader - if these seem like genuine and motivating challenges, please read on...


## Modelling & Simplifying Assumptions in this notebook
The *originating process*, T, will be modelled as emitting n events per time unit, where the mean per time unit follows a lognormal distribution (given the number of events emitted cannot be negative for a time interval).  The timestamps of those events per time unit will be entirely random, but rate-limited to a fixed service rate, α, events per second.

Each subsequent *process*, A, B, C etc. will be modelled as having 
- a processing latency following a normal distribution with standard parameters (μ, σ)
- a service rate of α tranactions per time unit, α, events per second
- a FIFO queue is assumed at the ingress of all processes with infinite possible queue depth
- while in reality processes are multi-threaded, this is not modelled explicitly, instead described by the processing latency and service rate

Each *event* will contain the following data points
- An event name/ID allowing the process generating the event to be uniquely identified.
- One or more Transaction References unique to the event chain that must be shared with at least one other event in the chain
- The Timestamp at which the transaction was processed, generating the event.


## Approach & Outcomes
This notebook will set out to assert:
1. It is possible to determine event chains via their related refences with O(1) complexity.  In other words, as we accumlate events and event chains, the time required to search for related references and build event chains does not increase.
2. Once individual event chains have been established, it is possible to determine the adjacency matrix of the processing tree by comparing the co-linearity of timestamps between processes, and by extension identify the originating and terminal nodes in the tree.
3. It is possible to determine both when queues have formed at the ingress of a process due to the arrival rate exceeding the service rate for that process
4. Once queuing has been accounted for, it is possible to determine the usual processing latency for each process
5. Finally, to determine:
- when queues are increasing and at which processes
- when the latency of a process changes (i.e. starts to follow a different distribution)
- the expected end-to-end processing time through the tree.  And by extension, any transactions that have yet to complete within this timeframe, or did complete processing through the tree but outside of the expected time range.


## Requirements for running this notebook
- Python 3.11 or above & ability to install packages from pypi
- Redis Stack 7.4.2 or above


## Terminology & Notation
- *Telemetry* - the remote measurement of a process, normally via second-order characteristics
- *Non-linear* - i.e. not in a strict singular sequence, a tree-like or network structure
- *Noisy* - not strictly predictable, i.e. non-deterministic
- *Processing latency* - the time taken for a transaction to pass through a process from the perspective of the process, i.e. not taking into account any queuing time before processing starts
- *Service rate* - the maximum throughput of a process (term taken from queuing theory)
- *Transaction* - a logical object that passess through a set of processes, denoted by capital T
- *Process* - a node in a graph of many processes, denoted by a capital letter, A, B, C, D, E
- *Event* - generated when a transaction passes through that particular process.  T1 would generate events A1, B1, C1 as it passes through A, B and C
- *Event Chain* = events belonging to the same transaction: A1 -> B1 -> C1
- *Event Set* = events belonging to the same process accross all transactions: An, Bn, Cn
- *Processing/Depedency Tree* = the generalised dependency tree that is true for all events in all event chains

In [2]:
#!pip install --upgrade numpy pandas scipy plotly ipycytoscape redis ulid
from datetime import datetime
import json
from io import StringIO

import numpy as np
import pandas as pd
import polars as pl
from scipy import stats
from plotly import graph_objects as go
import plotly.express as px
import redis
import ipycytoscape
import ulid

In [3]:
"""
Set the parameters of the simulation

*Processing Tree*
A -> B -> D
A -> C -> E -> F
A -> C -> E -> G
A -> H  (for 50% of events only)

*Latencies and Service Rates per second*
T: μ = 5,   σ = 8,   α = 28
A: μ = 2,   σ = 0.8, α = 25
B: μ = 0.9, σ = 0.3, α = 25 (initally, moving to μ = 2, σ = 2, α = 25)
C: μ = 0.9, σ = 0.3, α = 25
H: μ = 1.5, σ = 0.6, α = 12.5
D: μ = 2.5, σ = 1,   α = 28
E: μ = 2.5, σ = 1,   α = 28
F: μ = 0.9, σ = 0.2, α = 7
G: μ = 0.5, σ = 0.1, α = 12.5

*Transaction IDs*
A: An
B: An, Bn
C: Cn, An
D: Dn, Bn
E: En, Cn
F: Fn, Cn, En
G: Gn, En + one of An or Cn,
H: An

*Start/Duration*
19th March 2025 at midday
120 seconds / time intervals
"""

START_TIME = np.datetime64("2025-03-19T12:00:00.000")
NUM_INTERVALS = 10
T_MU = int(np.log(15)) #1.5
T_SIGMA = int(np.log(50)) #1.7
T_ALPHA = 75
proc_params = {"A": {"mu":0.5, "sigma":0.5, "alpha":60, "dep":"T", "refs":["A"], "context":["tea"]},
               "B": {"mu":0.9, "sigma":0.3, "alpha":25, "dep":"A", "refs":["A", "B"]},
               "C": {"mu":0.9, "sigma":0.3, "alpha":25, "dep":"A", "refs":["C", "A"], "context":["tea", "coffee"]},
               "D": {"mu":2.5, "sigma":1.0, "alpha":25, "dep":"B", "refs":["D", "B"]},
               "E": {"mu":2.5, "sigma":1.0, "alpha":12, "dep":"C", "refs":["E", "C"]},
               "F": {"mu":0.9, "sigma":0.2, "alpha":60, "dep":"E", "refs":["F", "C", "E"], "context":["milkshake"]},
               "G": {"mu":0.5, "sigma":0.1, "alpha":7, "dep":"E", "refs":["G", "E", ["A", "C"]]},
               "H": {"mu":1.5, "sigma":0.6, "alpha":25, "dep":"A", "refs":["A"]},
}

In [4]:
def apply_service_rate(timeseries: np.ndarray, rate_per_second: int) -> np.ndarray:
    """ Function to apply a service rate (maximum throughput rate to a timeseries.

        Inputs:
        timeseries - and numpy array of type np.datetime64
        rate_per_second = a scalar integer indicating the number of events per second

        Returns:
        The original timeseries now throttled by the service rate, 
        as a numpy array of type np.datetime64

        Steps:
        1) Convert the timeseries to Pandas in order to make it easier to record the original order
        in a column named 'index'
        2) Caclulate the time delta between events, increasing these to the rate limit interval where they are smaller
        3) Calculate the new time series based on the rate-limited cumulative time deltas and return timestamps
        in the original order (which may not have been sorted by timestamp)
    """
    timeseries = pd.DataFrame(timeseries, columns=["timeseries"]).sort_values("timeseries")
    timeseries["time_deltas_between_events"] = timeseries["timeseries"].diff()
    timeseries["rate_limited_time_deltas_between_events"] = np.maximum(timeseries["time_deltas_between_events"], 
                                                            np.full(timeseries["time_deltas_between_events"].shape, np.timedelta64(int(1/rate_per_second*1000),"ms")))
    timeseries.loc[timeseries["rate_limited_time_deltas_between_events"].isna(), "rate_limited_time_deltas_between_events"] = np.timedelta64(0, "ms")
    timeseries["rate_limited_timeseries"] = timeseries["rate_limited_time_deltas_between_events"].cumsum() + timeseries.loc[0,"timeseries"]
    return timeseries.sort_index()["rate_limited_timeseries"].to_numpy()

In [5]:
""" Simulate the originating event
"""
T = np.ndarray([0], dtype=np.datetime64)
for i in range(NUM_INTERVALS):
    #First determine how many originating events we will generate for this time interval
    #T_num_emissions_this_interval = min(int(np.random.lognormal(mean=np.log(T_MU), sigma=np.log(T_SIGMA))), T_ALPHA)
    T_num_emissions_this_interval = min(int(stats.lognorm(s=T_SIGMA, scale=np.exp(T_MU)).rvs(1)[0]), T_ALPHA)

    if T_num_emissions_this_interval > 0:
        #Generate the number of events from above at entirely random timestamps within this time interval & append to a list
        T_this_Interval = np.sort(np.random.random(T_num_emissions_this_interval).round(3)*1000).astype(np.timedelta64) + START_TIME + np.timedelta64(i, "s")
        T = np.concatenate((T, T_this_Interval), axis=0)

NUM_OBSERVATIONS = T.shape[0]
print(f"Overall number of originating transactions: {NUM_OBSERVATIONS}")

Overall number of originating transactions: 139


In [6]:
""" Simulate processing time in all processes, subject to their service rate
"""
timestamps = {}
timestamps["T"] = T
for proc, params in proc_params.items():
    processing_latency_this_process = (np.absolute(stats.norm(loc=params["mu"], scale=params["sigma"]).rvs(NUM_OBSERVATIONS).round(3))*1000).astype(np.timedelta64)
    timestamps[proc] = apply_service_rate(timestamps[params["dep"]], params["alpha"]) + processing_latency_this_process
del timestamps["T"]
NUM_EVENTS = len(timestamps) * len(timestamps["A"])
print(f"Overall number of events: {NUM_EVENTS}")

Overall number of events: 1112


In [7]:
""" Simulate transaction references
"""
trans_refs = {}
for proc, params in proc_params.items():
    refs_this_process = []
    for i in range(NUM_OBSERVATIONS):
        refs_this_event = []
        refs = params["refs"]
        for ref in refs:
            if type(ref) ==  str:
                refs_this_event.append({"type": ref, "id": ref + str(i), "ver": 1})
            if type(ref) == list:
                ref = np.random.choice(ref, 1, p=[1/len(ref)]*len(ref))
                refs_this_event.append({"type": str(ref[0]), "id": ref[0] + str(i), "ver": 1})
        refs_this_process.append(refs_this_event)
    trans_refs[proc] = refs_this_process

In [8]:
""" Simulate additional context information
"""
context = {}
for proc, params in proc_params.items():
    context_items_this_process = []
    for i in range(NUM_OBSERVATIONS):
        context_items_this_event = {}
        try:
            context_items = params["context"]
            for item in context_items:
                context_items_this_event[item]=i
        except:
            pass
        #if len(context_items_this_event) > 0:
        context_items_this_process.append(context_items_this_event)
    context[proc] = context_items_this_process

In [9]:
""" Load events into a Pandas DataFrame to facilitate ordering roughly in timestamp order
    with some local re-shuffling
    and export as json
"""
event_data = pd.DataFrame([{"EventName":k, "Timestamp":val, "Refs":trans_refs[k][i],"Context":context[k][i]} for k,v in timestamps.items() for i, val in enumerate(v)])
event_data = event_data.sort_values("Timestamp").reset_index()
event_data["index"] = event_data["index"] + np.random.randint(0,NUM_OBSERVATIONS, event_data.shape[0])
event_data.set_index("index").sort_index().reset_index(drop=True).to_json("eventdata.json", orient="records")
event_data.shape, event_data.set_index("index").sort_index().reset_index(drop=True).sample(5)

((1112, 5),
     EventName               Timestamp  \
 65          A 2025-03-19 12:00:10.024   
 464         D 2025-03-19 12:00:07.089   
 739         F 2025-03-19 12:00:18.268   
 306         C 2025-03-19 12:00:09.411   
 601         E 2025-03-19 12:00:19.869   
 
                                                   Refs  \
 65              [{'type': 'A', 'id': 'A94', 'ver': 1}]   
 464  [{'type': 'D', 'id': 'D16', 'ver': 1}, {'type'...   
 739  [{'type': 'F', 'id': 'F85', 'ver': 1}, {'type'...   
 306  [{'type': 'C', 'id': 'C62', 'ver': 1}, {'type'...   
 601  [{'type': 'E', 'id': 'E107', 'ver': 1}, {'type...   
 
                        Context  
 65                 {'tea': 94}  
 464                         {}  
 739          {'milkshake': 85}  
 306  {'tea': 62, 'coffee': 62}  
 601                         {}  )

### A quick Redis Cribsheet
- Redis Default Connection URL: localhost:6379
- Redis Insight: http://localhost:8001/redis-stack/browser

#### Redis Commands
- FLUSHDB - delete the full contents of Redis including indices

#### Useful python functions for redis:
Converting to/from unix timestamp
- datetime.fromtimestamp(1610668800)
- datetime(2021,1,5, 0, 0).timestamp()

JSONPath cribsheet:
- $ - the root object or element
- @ - current object or element
- . - child operator, used to denote a child element of the current element
- .. - recursive scan
- * - wildcard, returning all objects or elements regardless of their names
- [] - subscript operator / array operator
- , - union operator, returns the union of the children or indexes indicated
- : - array slice operator; you can slice arrays using the syntax [start:step]
- () - lets you pass a script expression in the underlying implementation’s script language
- ?()	- applies a filter/script expression to query all items that meet certain criteria

In [10]:
"""
Step #1 - grouping events into their event chain

In order to be able to correlate an incoming event via the transaction references, we need to know 2 things:
1) event chains that contain any of the set of transaction reference on the event being ingested
2) if the related refs on those event chains intersect or conflict with those on the incoming event.  There can only be a single reference of a given type in an event chain

Aglorithm:
- If no event chains found containing the incoming refs, create a new event chain with those refs
- If > 0 event chains found containing the incoming refs, with mutually distinct ref types from those refs not found on the incoming event, then append refs and the incoming event to those event chains
- If > 0 found and event types overlap/conflict with those on the incoming event, then duplicate the chain ensuring there is only one ref type per event chain

We can see 2 possible mechanisms:
- 1. Maintain hashes by transaction ID of eventchain and related reference and types, or
- 2. Maintain a single object - the event chain - that contains related references and types
"""

# INCOMPLETE SOLUTION USING REDIS OM - included here to facilitate understanding the data model
# This solution uses Redis JSON via Redis OM which will create a connection using default settings
from redis_om import Field, JsonModel, EmbeddedJsonModel, Migrator
from pydantic import PositiveInt
from typing import List

# This class models the embedded array of ref object
class Ref(EmbeddedJsonModel):
    reftype: str
    refid: str = Field(index=True)
    refver: PositiveInt

# This class models the embedded array of eventtimestamp objects.
class EventTimestamp(EmbeddedJsonModel):
    process: str
    eventtimestamp: datetime

# This class models an event chain.
class EventChain(JsonModel):
    #refs: List[Ref] = Field(index=True)
    concatenatedrefs: List[str] = Field(index=True)
    eventtimestamps: List[EventTimestamp]
    complete: bool

    # Config to specify how to generate key names
    class Meta:
        global_key_prefix="argus"
        model_key_prefix="eventchain"

#create a dummy eventchain object & search index
dummy_eventchain = {"concatenatedrefs": ["dummytype1_dummuid1_1",
                                         "dummytype2_dummuid2_2"
                                        ],
                    "eventtimestamps": [{"process": "procA", "eventtimestamp": 1610668800},
                                        {"process": "procB", "eventtimestamp": 1610668801}
                                       ] 
                    }

dummy_event = EventChain(**dummy_eventchain)
dummy_event.save()
Migrator().run()

ModuleNotFoundError: No module named 'redis_om'

In [ ]:
#WORKING EXAMPLE *not* using the ORM
from redis.commands.search.field import TagField
from redis.commands.search.indexDefinition import IndexDefinition, IndexType

#First create an index on the concatenated ref
REDIS_KEY_BASE = "argus:ec"
INDEX_NAME = "argus:ec:idx"
STREAM_NAME = "argus:ecstream"
pool = redis.ConnectionPool(host="localhost", port=6379, decode_responses=True)
r = redis.Redis(connection_pool=pool)
try:
    r.ft(INDEX_NAME).dropindex()
except:
    pass
r.ft(INDEX_NAME).create_index(
    [
        TagField("$.concatenatedrefs[*]", as_name="concatenatedrefs"),
        TagField("$.complete", as_name="complete")
    ], 
    definition=IndexDefinition(
        index_type=IndexType.JSON, 
        prefix=[f"{REDIS_KEY_BASE}:"]
    )
)

#Function to return the key of the eventchain objects
def create_key():
    key = str(ulid.ulid())
    return f"{REDIS_KEY_BASE}:{key}"

In [ ]:
 #Load our simulated events into our application
with open("eventdata.json") as f:
    events = json.load(f)
print(f"{len(events)} events loaded")

608 events loaded


In [ ]:
#Create an empty adjacency matrix - here it will be empty, 
#but later we will populate, and we can see the effect on the search
adjacency_matrix = pd.DataFrame()


In [ ]:
#Load the expected, source and terminal events
#This can be re-run a 2nd time after the adjacency matrix is created later in this notebook in order to see the effect
if len(adjacency_matrix) > 0:
    expected_events = set(adjacency_matrix["source"]) | set(adjacency_matrix["target"])
    source_nodex = set(adjacency_matrix["source"]) - set(adjacency_matrix["target"])
    terminal_nodes = set(adjacency_matrix["target"]) - set(adjacency_matrix["source"]) 
else:
    expected_events = set()
    source_nodex = set()
    terminal_nodes = set()

In [ ]:
def add_or_merge_event_to_event_chain(r, event):
    """ This function will search on the given redis connection for an existing event chain
        with any of the references in the given event.  It will create a new event chain or update
        existing event chains accordingly.  Finally it will push the created/update event chain id to a stream
    """

    #Workaround - we need to find event chains with tuples of (reftype, ref, id)
    #Here for ease we concatenate those three values to a string separated with underscore characters
    #TODO - look at alternate methods
    #Further it could be that an event itself contains multiple refs of the same type
    #TODO - in that case it would be necessary to split/duplicate the event and treat as 2 events
    #A bloom filter might be more performant in order to determine if any of the IDs have been seen before
    #TODO - experiment with a Bloomfilter - this exists in Redis Stack, and Python implementation of a 
    #Bloomfilter is below.  We also have a BF coded in Java already in our Next Gen DB
    event_concatenatedrefs = ["_".join([str(i) for i in ref.values()]) for ref in event["Refs"]]
    query_string = "@concatenatedrefs:{" + "|".join(event_concatenatedrefs) + "} "   
    query_string += "@complete:{false}"
    results = r.ft("argus:ec:idx").search(query_string)
    
    #TODO Multi-thread at this point based on chain ID modulo the number of threads/processes
    event_timestamp = {event["EventName"]:event["Timestamp"]}
    context = event["Context"]

    if len(results.docs) == 0:
        #create a new event chain
        j = {"concatenatedrefs": event_concatenatedrefs,
             "timestamps":event_timestamp,
             "context": context,
             "complete":False}                    
        chain_id = create_key()
        p = r.pipeline()
        p.json().set(chain_id, "$", j)
        p.xadd(STREAM_NAME, {"ecid": chain_id})
        p.execute()
        pass

    #TODO The code below is quite repetative - re-org/re-factoring opportunity!   
    else:
        p = r.pipeline()
        #if the references on the results are mutually exclusive, then merge those to a single document
        if len(results.docs) > 1:
            for i in range(len(results.docs)):
                for j in range(i+1, results.docs):
                    #TO DO
                    pass

        for result in results.docs:
            #store the id for this chain - we'll need to update it
            #get a full list of refs already on the chain
            #and workout those that are missing
            #finally get the process name and timestamp of this event
            chain_json_this_result = json.load(StringIO(result.json))
            refs_this_result = chain_json_this_result["concatenatedrefs"]
            events_this_result = chain_json_this_result["timestamps"].keys()
            chain_id_this_result = result.id
            refs_not_found = set(event_concatenatedrefs) - set(refs_this_result)

            if len(refs_not_found) == 0:
                #if all of the refs are already present, then it is just matter of adding this event to the chain
                #p.json().arrappend(chain_id_this_result, "$.timestamps", event_timestamp)
                p.json().merge(chain_id_this_result, "$.timestamps", event_timestamp)
                p.json().merge(chain_id_this_result, "$.context", context)


            elif len(refs_not_found) > 0:
                #work out if there is any potential conflict - we only want one ID type per chain
                idtypes_in_conflict = {ref.split("_")[0] for ref in refs_this_result} & {ref.split("_")[0] for ref in refs_not_found}                

                #if no conflicts then we can simply add this event & missing refs to the chain
                if len(idtypes_in_conflict) == 0:
                    #p.json().arrappend(chain_id_this_result, "$.timestamps", event_timestamp)
                    p.json().merge(chain_id_this_result, "$.timestamps", event_timestamp)
                    p.json().merge(chain_id_this_result, "$.context", context)
                    for ref in refs_not_found:
                        p.json().arrappend(chain_id_this_result, "$.concatenatedrefs", ref)

                #if conflicts: add the event to the chain
                #add the refs not in conflict to the chain
                #duplicate the chain
                #add the refs in conflict separately and individually to the chains
                else:
                    refs_not_in_conflict = [str(r) for r in refs_not_found if r.split("_")[0] not in idtypes_in_conflict]
                    refs_in_conflict = [str(r) for r in refs_not_found if r.split("_")[0] in idtypes_in_conflict]
                    refs_minus_conflicts = [r for r in refs_this_result if r.split("_")[0] not in idtypes_in_conflict]
                    #add to the existing chain
                    #p.json().arrappend(chain_id_this_result, "$.timestamps", event_timestamp)
                    p.json().merge(chain_id_this_result, "$.timestamps", event_timestamp)
                    p.json().merge(chain_id_this_result, "$.context", context)
                    for r in refs_not_in_conflict:
                        p.json().arrappend(chain_id_this_result, "$.concatenatedrefs", r)
                    #create a 2nd chain
                    chain_json_this_result["timestamps"].append(event_timestamp)
                    chain_json_this_result["concatenatedrefs"] = refs_minus_conflicts + refs_in_conflict
                    p.json().set(create_key(), "$", chain_json_this_result)           

            awaited_events = expected_events - set(events_this_result) - set(event_timestamp.keys())
            if set(event_timestamp.keys()) in expected_events and len(awaited_events) == 0:
                p.json().toggle(chain_id_this_result, "$.complete")
            p.xadd(STREAM_NAME, {"ecid": chain_id_this_result}, maxlen=1000000, approximate=True)

        p.execute()
    
for event in events:
    add_or_merge_event_to_event_chain(r, event)

In [ ]:
#Read the event chain from the stream for processing / monitoring / alerting
#The most recent x amount of time can be read from the stream 
lookbackperiod_ms = 5 * 60 * 1000   #i.e. 5 mins
start_timestamp = int(datetime.now().timestamp()) * 1000 - lookbackperiod_ms
results = r.xrange(name=STREAM_NAME, min=str(start_timestamp)+"-0", max="+")
event_chain_keys = [r[1]["ecid"] for r in results]
if len(event_chain_keys) > 0:
    event_chains = r.json().mget(event_chain_keys, "$")
event_chains = [i[0] for i in event_chains]
print(f"fetched {len(event_chain_keys)} event chains")

#view latencies from the stream including the combined context etc.
from_event = "A"
to_event = "B"
pd.DataFrame(pd.DataFrame(event_chains)["timestamps"].apply(lambda x: datetime.fromtimestamp((x[to_event] - x[from_event])/1000)).dt.time.rename(f"Latency from {from_event} to {to_event}")).merge(
            pd.DataFrame(pd.DataFrame(event_chains)["context"].to_list()),\
            how="left", left_index=True, right_index=True)

fetched 608 event chains


,Latency from A to B,tea,coffee,milkshake
0,00:00:00.864000,12,12,12
1,00:00:01.480000,3,3,3
2,00:00:01.523000,1,1,1
3,00:00:00.740000,5,5,5
4,00:00:01.019000,0,0,0
...,...,...,...,...
603,00:00:01.866000,72,72,72
604,00:00:01.617000,56,56,56
605,00:00:01.351000,74,74,74
606,00:00:00.985000,69,69,69


In [ ]:
""" Step #2 Populate the Adjacency Matrix
"""
#first get the keys of the event chains we'd like to analyse
#here we simply get all of our toy data, but equally there could be an
#index on timestamp, or some other indexed attribute we could use to retrieve
#a sample population of event chains
keys = r.keys("argus:ec:*")
eventchain_timestamps = r.json().mget(keys, "$.timestamps")

#load into a DataFrame, sort by the column that has the lowest value
eventchain_timestamps = pd.DataFrame([d[0] for d in eventchain_timestamps])
minvals = eventchain_timestamps.min()
min_col = minvals.index[minvals.argmin()]
eventchain_timestamps = eventchain_timestamps.sort_values(min_col).reset_index(drop=True)

#Visualise the time series
fig = px.line(eventchain_timestamps.reset_index().melt("index"), x="index", y="value", color="variable")
fig.show()

In [ ]:
""" Now calculate the co-linearity and infer the dependency tree
"""

#Set a maximum pvalue for the Pearson correlation to be considered relevant
max_pval = 0.05

#Now we see if we can reverse-engineer the sequence of dependencies, purely from the timestamps of the event collections

#Create a matrix of all possible correlations, initialised to 0
#Rows correlate to the event, colums the dependency

number_sets  = len(eventchain_timestamps.columns)
event_sets = eventchain_timestamps.T.to_numpy()
event_labels = eventchain_timestamps.columns.to_list()
corr = np.zeros((number_sets, number_sets))

for e in range(number_sets):
    for d in range(number_sets):

        #create an index of only those items which are not NAN in both the event and potential dependency.
        index = ~np.isnan(event_sets[e]) & ~np.isnan(event_sets[d])

        #an event cannot be dependent on itself
        if e == d:  
            p = 0
        #test that the event is later than the potential dependency in all cases
        elif (event_sets[e][index] >= event_sets[d][index]).all():          #TO DO - calculate the deltas, throw away z-score of s>2 and then see if all positive
            r = stats.pearsonr(event_sets[e][index], event_sets[d][index], alternative="greater")
            #the pearsonr function returns a tuple, [0] is the correlation co-efficient rho, [1] is the pvalue of the null hypothesis
            if r[1] < max_pval:     #if the pval is less than the max pval set above, then accept the pearson correlation
                p = r[0]
            else:
                p=0
        #events are not later than the potential dependency, therefore the correlation is zero
        else:
            p = 0
        #update the correlation matrix
        corr[e, d] = p

print(corr)

nodes = []
edges = []

#print the results in an understandable way
for e in range(number_sets):

    nodes.append({"data": {"id": event_labels[e], "label": event_labels[e]}})

    if (corr[e]==0).all():
        print("Event {e} is a root event".format(e=event_labels[e]))
    else:
        d = np.argmax(corr[e])

        index = ~np.isnan(event_sets[e]) & ~np.isnan(event_sets[d])

        delta = (event_sets[e][index] - event_sets[d][index])
        mean_delta = np.round(np.mean(delta),3)
        std_delta = np.round(np.std(delta), 3)
        max_delta = np.round(np.max(delta), 3)
        min_delta = np.round(np.min(delta))

        label = "m={mean_delta}\n s={std_delta}\n max={max_delta}\n min={min_delta}".format(mean_delta=mean_delta, std_delta=std_delta, max_delta=max_delta, min_delta=min_delta)

        print("Event {e} is likely to be dependent on {d}".format(e=event_labels[e], d=event_labels[d]), label)

        edges.append({'data': {'source': event_labels[d], 'target': event_labels[e], "label": label,  "isdirected": True}})

adjacency_matrix = pd.DataFrame([(i["data"]["source"], i["data"]["target"] )for i in edges], columns=["source","target"])

"""
Expected Result:
A -> B -> D
A -> C -> E -> F
A -> C -> E -> G
"""


[[0.         0.         0.         0.         0.         0.
  0.         0.        ]
 [0.9965023  0.         0.         0.         0.         0.
  0.         0.        ]
 [0.99602803 0.         0.         0.         0.         0.
  0.         0.        ]
 [0.96905231 0.97371154 0.96467567 0.         0.         0.
  0.         0.        ]
 [0.96327128 0.96243662 0.96954588 0.         0.         0.
  0.         0.        ]
 [0.96274762 0.96160786 0.96856635 0.         0.99900565 0.
  0.         0.95326715]
 [0.95910805 0.95907978 0.9663897  0.         0.99918496 0.
  0.         0.94882332]
 [0.98711059 0.         0.         0.         0.         0.
  0.         0.        ]]
Event A is a root event
Event B is likely to be dependent on A m=1219.645
 s=321.729
 max=2138
 min=622
Event C is likely to be dependent on A m=1215.487
 s=380.027
 max=2110
 min=337
Event D is likely to be dependent on B m=2750.737
 s=866.249
 max=4276
 min=1104
Event E is likely to be dependent on C m=3863.026
 s=1

'\nExpected Result:\nA -> B -> D\nA -> C -> E -> F\nA -> C -> E -> G\n'

In [ ]:
#Visualise the inferred dependency tree as a directed acylic graph
elements = {"nodes": nodes, 
            "edges": edges}

cytoscapeobj = ipycytoscape.CytoscapeWidget()
cytoscapeobj.graph.add_graph_from_json(elements, directed=True)
cytoscapeobj.set_layout(name="dagre")   #dagre, klay
cytoscapeobj.set_style([{
                          "selector":"node",
                          "style":{
                             "label-position": "right",
                             "font-family": "arial",
                             "font-size": "12px",
                             "label": "data(label)"
                          }
                       },
                       {
                          "selector":"edge",
                          "style":{
                             "width": 4,
                             "line-color": "#888",
                             "target-arrow-color": "#888",
                             "target-arrow-shape": "triangle",
                             "curve-style": "bezier",
                             "label": "data(label)",
                             "text-wrap": "wrap",
                             "font-family": "arial",
                             "font-size": "8px",
                          }
                       },
                       
                    ])


cytoscapeobj

CytoscapeWidget(cytoscape_layout={'name': 'dagre'}, cytoscape_style=[{'selector': 'node', 'style': {'label-pos…

### WIP FROM THIS POINT ONWARDS
### Queuing Crib Sheet

- λ = arrival rate
- α = service rate
- ρ = traffic intensity = λ / α
- W = average time a customer spends in the system
- L = averge number of customers in the system = λW
- Lq = average number of customers in the queue except those being served = ρ / (1-ρ)



In [ ]:
NUM_CHAINS = eventchain_timestamps.shape[0]
cum_observation_count = (np.arange(NUM_CHAINS) + 1)
alpha = pd.DataFrame(index=eventchain_timestamps.index)
lambd = pd.DataFrame(index=eventchain_timestamps.index)
rho = pd.DataFrame(index=eventchain_timestamps.index)
print(f"Number of chains: {NUM_CHAINS}")

In [ ]:
for col in eventchain_timestamps.columns:
    #Alpha average service rate per second for the whole period sampled.  TODO think about having a sampling window
    minval = eventchain_timestamps[col].min()
    alpha[col] = (cum_observation_count / (eventchain_timestamps[col].sort_values() - minval).T * 1000).sort_index()
    alpha = alpha.replace(np.inf, 0)
    #Lambda is the Alpha of the source
    adj = adjacency_matrix.loc[adjacency_matrix["target"]==col, "source"]
    if len(adj) > 0:
        source = adj.to_list()[0]
    else:
        source = col
    lambd[col] = alpha[source]
    rho = lambd / alpha
    Lq = rho / (1 - rho)

#alpha.to_csv("alpha.csv")
alpha

In [ ]:
""" Python implementation of a Bloom Filter
    taken from https://www.geeksforgeeks.org/bloom-filters-introduction-and-python-implementation/
    Here, provided for reference as we will be using Redis built-in Bloom Filter function
"""
# Python 3 program to build Bloom Filter
# Install mmh3 and bitarray 3rd party module first
# pip install --upgrade mmh3 bitarray
import math
import mmh3
from bitarray import bitarray


class BloomFilter(object):

    '''
    Class for Bloom filter, using murmur3 hash function
    '''

    def __init__(self, items_count, fp_prob):
        '''
        items_count : int
            Number of items expected to be stored in bloom filter
        fp_prob : float
            False Positive probability in decimal
        '''
        # False possible probability in decimal
        self.fp_prob = fp_prob

        # Size of bit array to use
        self.size = self.get_size(items_count, fp_prob)

        # number of hash functions to use
        self.hash_count = self.get_hash_count(self.size, items_count)

        # Bit array of given size
        self.bit_array = bitarray(self.size)

        # initialize all bits as 0
        self.bit_array.setall(0)

    def add(self, item):
        '''
        Add an item in the filter
        '''
        digests = []
        for i in range(self.hash_count):

            # create digest for given item.
            # i work as seed to mmh3.hash() function
            # With different seed, digest created is different
            digest = mmh3.hash(item, i) % self.size
            digests.append(digest)

            # set the bit True in bit_array
            self.bit_array[digest] = True

    def check(self, item):
        '''
        Check for existence of an item in filter
        '''
        for i in range(self.hash_count):
            digest = mmh3.hash(item, i) % self.size
            if self.bit_array[digest] == False:

                # if any of bit is False then,its not present
                # in filter
                # else there is probability that it exist
                return False
        return True

    @classmethod
    def get_size(self, n, p):
        '''
        Return the size of bit array(m) to used using
        following formula
        m = -(n * lg(p)) / (lg(2)^2)
        n : int
            number of items expected to be stored in filter
        p : float
            False Positive probability in decimal
        '''
        m = -(n * math.log(p))/(math.log(2)**2)
        return int(m)

    @classmethod
    def get_hash_count(self, m, n):
        '''
        Return the hash function(k) to be used using
        following formula
        k = (m/n) * lg(2)

        m : int
            size of bit array
        n : int
            number of items expected to be stored in filter
        '''
        k = (m/n) * math.log(2)
        return int(k)
